# 8.4 Bag-of-words & TF-IDF

Bag-of-words deliberately forgets order, then TF-IDF asks which surviving words are common enough to trust and rare enough to matter.

The document-term matrix fixes vocabulary axes. TF-IDF then weights local term frequency by smoothed inverse document frequency so rare but reliable terms influence retrieval more than ubiquitous words.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): a fixed-vocabulary TF-IDF matrix

For document $d$ and term $t$, the lesson uses $$w_{d,t}=\operatorname{tf}_{d,t}\left(\log\frac{1+N}{1+\operatorname{df}_t}+1\right).$$ The D1 corpus is `cat sat`, `cat ate fish`, `dog sat`, with sorted vocabulary $[\textsf{ate},\textsf{cat},\textsf{dog},\textsf{fish},\textsf{sat}]$.

In [ ]:

def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def count_matrix(docs, vocabulary=None):
    tokenized = [tokenize_doc(doc) for doc in docs]
    if vocabulary is None:
        vocabulary = sorted({token for doc in tokenized for token in doc})
    index = {term: i for i, term in enumerate(vocabulary)}
    matrix = np.zeros((len(docs), len(vocabulary)), dtype=float)
    for row, tokens in enumerate(tokenized):
        counts = Counter(tokens)
        for term, count in counts.items():
            if term in index:
                matrix[row, index[term]] = count
    return matrix, vocabulary, tokenized

lesson_docs = ["cat sat", "cat ate fish", "dog sat"]
lesson_counts, lesson_vocab, lesson_tokens = count_matrix(lesson_docs)
expected_counts = np.array([
    [0, 1, 0, 0, 1],
    [1, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
], dtype=float)

assert lesson_vocab == ["ate", "cat", "dog", "fish", "sat"]
assert lesson_counts.shape == (3, 5)
assert np.array_equal(lesson_counts, expected_counts)
assert int((lesson_counts > 0).sum()) == 7


The reusable method freezes the vocabulary, computes term frequency per document, applies the lesson's smoothed IDF, and supports cosine retrieval.

In [ ]:

def tfidf_matrix(docs, vocabulary=None):
    counts, vocabulary, tokenized = count_matrix(docs, vocabulary)
    lengths = np.maximum(1, counts.sum(axis=1, keepdims=True))
    tf = counts / lengths
    df = (counts > 0).sum(axis=0)
    n_docs = counts.shape[0]
    idf = np.log((1 + n_docs) / (1 + df)) + 1
    weights = tf * idf
    return weights, counts, vocabulary, idf


def retrieve(query, docs, vocabulary, labels=None):
    doc_weights, doc_counts, vocab, idf = tfidf_matrix(docs, vocabulary)
    query_weights, query_counts, vocab, query_idf = tfidf_matrix([query], vocabulary)
    scores = [cosine(query_weights[0], row) for row in doc_weights]
    best = int(np.argmax(scores))
    if labels is None:
        return best, scores
    return labels[best], scores

lesson_weights, lesson_counts, lesson_vocab, lesson_idf = tfidf_matrix(lesson_docs)
cat_doc_weights, cat_counts, cat_vocab, cat_idf = tfidf_matrix(["cat cat sat"], lesson_vocab)
cat_tf = cat_counts[0, lesson_vocab.index("cat")] / cat_counts.sum()

assert abs(cat_tf - (2 / 3)) < 1e-12
assert abs(lesson_idf[lesson_vocab.index("cat")] - (math.log(4 / 3) + 1)) < 1e-12


## Dataset ladder: inline documents from toy corpus to longer mixed-topic documents

In [ ]:

bow_ladder = [
    {
        "name": "D1 lesson cat sat corpus",
        "docs": ["cat sat", "cat ate fish", "dog sat"],
        "labels": ["cat", "cat", "dog"],
        "queries": [("cat sat", "cat")],
    },
    {
        "name": "D2 five clean topic docs",
        "docs": ["ads campaign click", "ads budget bid", "invoice payment receipt", "creative image asset", "creator video content"],
        "labels": ["ads", "ads", "billing", "creative", "creator"],
        "queries": [("campaign bid", "ads"), ("payment receipt", "billing"), ("video creator", "creator")],
    },
    {
        "name": "D3 negation and order collisions",
        "docs": ["good product", "not good product", "refund approved", "refund not approved", "delivery fast"],
        "labels": ["positive", "negative", "positive", "negative", "positive"],
        "queries": [("not good", "negative"), ("refund approved", "positive"), ("not approved", "negative")],
    },
    {
        "name": "D4 tiny inline news snippets",
        "docs": ["market rally after earnings", "sports team wins final", "weather storm delays flights", "earnings report lifts stock", "team signs creator sponsor"],
        "labels": ["finance", "sports", "weather", "finance", "sports"],
        "queries": [("stock earnings", "finance"), ("team final", "sports"), ("storm flights", "weather")],
    },
    {
        "name": "D5 longer mixed-topic documents",
        "docs": [
            "campaign manager reports creative click growth after budget change",
            "invoice receipt says payment failed and refund not approved",
            "creator marketplace video content earns sponsor revenue",
            "weather alert delays creator travel and campaign filming",
            "not good creative feedback asks for image asset revision",
        ],
        "labels": ["ads", "billing", "creator", "weather", "creative"],
        "queries": [("creative click budget", "ads"), ("refund not approved", "billing"), ("creator sponsor video", "creator"), ("not good image", "creative")],
    },
]

for rung in bow_ladder:
    print(rung["name"], "docs", len(rung["docs"]), "queries", len(rung["queries"]))
    print("sample", rung["docs"][0])


In [ ]:

bow_results = []
for index, rung in enumerate(bow_ladder, start=1):
    counts, vocabulary, tokenized = count_matrix(rung["docs"])
    predictions = []
    expected = []
    for query, gold in rung["queries"]:
        pred, scores = retrieve(query, rung["docs"], vocabulary, rung["labels"])
        predictions.append(pred)
        expected.append(gold)
    acc = exact_accuracy(predictions, expected)
    bow_results.append({"rung": index, "name": rung["name"], "accuracy": acc, "vocab": vocabulary})

for row in bow_results:
    print(row["rung"], row["name"], f"accuracy={row['accuracy']:.3f}", "vocab", len(row["vocab"]))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, rung in enumerate(bow_ladder):
    weights, counts, vocabulary, idf = tfidf_matrix(rung["docs"])
    preview = weights[: min(5, weights.shape[0]), : min(8, weights.shape[1])]
    show_matrix(axes[0, idx], preview, f"D{idx + 1} TF-IDF", vocabulary[: preview.shape[1]], rung["labels"][: preview.shape[0]])

x = [row["rung"] for row in bow_results]
y = [row["accuracy"] for row in bow_results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("Accuracy vs document complexity")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("retrieval accuracy")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: vocabulary mismatch and order-blind negation

In [ ]:

def add_bigrams(doc):
    tokens = tokenize_doc(doc)
    bigrams = [tokens[i] + "_" + tokens[i + 1] for i in range(len(tokens) - 1)]
    return " ".join(tokens + bigrams)

d5 = bow_ladder[-1]
frozen_counts, frozen_vocab, tokenized = count_matrix(d5["docs"])
query = "not good image"
wrong_vocab = sorted(set(tokenize_doc(query)))
wrong_pred, wrong_scores = retrieve(query, d5["docs"], wrong_vocab, d5["labels"])
fixed_docs = [add_bigrams(doc) for doc in d5["docs"]]
fixed_query = add_bigrams(query)
fixed_counts, fixed_vocab, fixed_tokenized = count_matrix(fixed_docs)
fixed_pred, fixed_scores = retrieve(fixed_query, fixed_docs, fixed_vocab, d5["labels"])

print("wrong vocab prediction", wrong_pred)
print("fixed bigram prediction", fixed_pred)
print("fixed has not_good", "not_good" in fixed_vocab)


## Evaluate it + Practice

- Metric: retrieval or classification accuracy with TF-IDF cosine
- No-skill baseline: raw count overlap without IDF or document-length normalization
- Cheap sanity check: the lesson D1 matrix must be shape 3 by 5 with 7 active binary entries
- Ablation: serve with a new vocabulary or remove bigrams and D3/D5 negation errors appear
- Failure signals: vectors of different lengths, long documents dominating, or no feature for not_good

### Practice

**Practice:** Add a common word to every D2 document and inspect its IDF.

**Practice:** Compare unigram-only and unigram-plus-bigram retrieval on the D3 negation examples.